In [7]:
# 可在 Jupyter Notebook 里直接运行

from __future__ import annotations
from pathlib import Path
import os
import fnmatch
from typing import Iterable, List, Tuple, Optional, Callable

def _should_ignore(name: str, ignore_globs: Optional[List[str]]) -> bool:
    if not ignore_globs:
        return False
    return any(fnmatch.fnmatch(name, pat) for pat in ignore_globs)

def _sort_key(p: Path) -> Tuple[int, str]:
    # 目录优先，再按名称不区分大小写排序
    return (0 if p.is_dir() else 1, p.name.lower())

def _children(
    p: Path,
    show_files: bool,
    follow_symlinks: bool,
    ignore_globs: Optional[List[str]],
) -> List[Path]:
    try:
        items = list(p.iterdir())
    except PermissionError:
        return []

    # 过滤隐藏/忽略项
    filtered = []
    for q in items:
        name = q.name
        if name.startswith(".") and not show_files:
            # 如果完全不想显示隐藏文件/夹，可以把 next 行替换为: continue
            pass
        if _should_ignore(name, ignore_globs):
            continue
        if q.is_dir() or show_files:
            filtered.append(q)

    # 如果不跟随符号链接目录，则把符号链接目录当作叶子
    if not follow_symlinks:
        out = []
        for q in filtered:
            if q.is_dir() and q.is_symlink():
                # 仍然显示这个链接，但不深入
                out.append(q)
            else:
                out.append(q)
        filtered = out

    return sorted(filtered, key=_sort_key)

def print_tree(
    root: str | Path,
    *,
    show_files: bool = True,
    max_depth: Optional[int] = None,
    ignore_globs: Optional[List[str]] = None,
    follow_symlinks: bool = False,
    prefix: str = "",
) -> None:
    """
    打印文件树结构。

    参数
    ----
    root : 根目录路径
    show_files : 是否显示文件（为 False 则只显示目录）
    max_depth : 最大深度（None 表示不限制；0 表示只显示 root 本身）
    ignore_globs : 忽略的通配符列表，例如 ["*.pyc", ".git", "__pycache__"]
    follow_symlinks : 是否深入符号链接目录
    prefix : 递归时内部使用的绘制前缀
    """
    root = Path(root)
    if max_depth is not None and max_depth < 0:
        return

    # 根行
    if prefix == "":
        print(root.resolve().as_posix())

    children = _children(root, show_files=show_files, follow_symlinks=follow_symlinks, ignore_globs=ignore_globs)
    # 仅保留目录（当 show_files=False）或全部
    if not show_files:
        children = [c for c in children if c.is_dir()]

    # 分支绘制
    for i, child in enumerate(children):
        connector = "└── " if i == len(children) - 1 else "├── "
        print(prefix + connector + child.name)

        # 递归
        if child.is_dir() and (follow_symlinks or not child.is_symlink()):
            if max_depth is None or max_depth > 0:
                extension = "    " if i == len(children) - 1 else "│   "
                print_tree(
                    child,
                    show_files=show_files,
                    max_depth=(None if max_depth is None else max_depth - 1),
                    ignore_globs=ignore_globs,
                    follow_symlinks=follow_symlinks,
                    prefix=prefix + extension,
                )

print_tree("/blue/shiboli.fsu/yifeisun.umich/adversarial_robustness_FNO/2D_NS_FNO2d_recurrent/saved_models/", 
           show_files=True, max_depth=None, 
           ignore_globs=["*.gif","*.txt","*.png","*.ipynb","*.pkl","*.err","*.out","*.csv","*.sh","*.py"])


/blue/shiboli.fsu/yifeisun.umich/adversarial_robustness_FNO/2D_NS_FNO2d_recurrent/saved_models
└── 2D
    ├── modes128_width40_epochs500_Tin10_T10
    │   └── NS_2d_FNO_model_trainedby_dim2d_nx256_N1150_solver=exponax_nu0.000_t20.0_train_all_frames.pth
    ├── modes12_width20_epochs500_Tin10_T10
    │   ├── gifs_test
    │   ├── gifs_train
    │   └── NS_2d_FNO_model_trainedby_dim2d_nx256_N1150_solver=exponax_nu0.000_t20.0_train_all_frames.pth
    ├── modes16_width60_epochs500_Tin10_T10
    │   └── NS_2d_FNO_model_trainedby_dim2d_nx256_N1150_solver=exponax_nu0.000_t20.0_train_all_frames.pth
    ├── modes32_width40_epochs500_Tin10_T10
    │   ├── gifs_test
    │   ├── gifs_train
    │   └── NS_2d_FNO_model_trainedby_dim2d_nx256_N1150_solver=exponax_nu0.000_t20.0_train_all_frames.pth
    ├── modes32_width40_epochs500_Tin1_T19
    │   └── NS_2d_FNO_model_trainedby_dim2d_nx256_N1150_solver=exponax_nu0.000_t20.0_train_all_frames.pth
    ├── modes32_width60_epochs500_Tin10_T10
    │   └── NS

In [8]:
from __future__ import annotations
from pathlib import Path
from typing import Optional, List, Tuple
import fnmatch

def _matches_any(name_or_rel: str, patterns: Optional[List[str]]) -> bool:
    if not patterns:
        return False
    return any(fnmatch.fnmatch(name_or_rel, pat) for pat in patterns)

def _should_ignore(p: Path, root: Path, ignore_globs: Optional[List[str]]) -> bool:
    """同时支持按 basename 和相对 root 的路径模式忽略。"""
    if not ignore_globs:
        return False
    name = p.name
    rel  = p.relative_to(root).as_posix() if p != root else p.name
    return _matches_any(name, ignore_globs) or _matches_any(rel, ignore_globs)

def _sorted_children(p: Path) -> List[Path]:
    try:
        items = list(p.iterdir())
    except PermissionError:
        return []
    # 目录优先，再按名称排序
    return sorted(items, key=lambda x: (0 if x.is_dir() else 1, x.name.lower()))

def print_tree_only_pth(
    root: str | Path,
    *,
    show_files: bool = True,                 # 对于只保留 .pth 叶子，设 True 更合理
    max_depth: Optional[int] = None,         # None 不限；0 只检查当前节点（若是 .pth 文件）
    ignore_globs: Optional[List[str]] = None,
    follow_symlinks: bool = False,
) -> None:
    """
    只打印那些最终叶节点是 .pth 文件的分支；不通向 .pth 的分支完全不打印。
    """

    root = Path(root)

    # 预先做一次“是否含 .pth 叶子”的判定，然后按需打印
    def contains_pth(node: Path, depth_left: Optional[int]) -> bool:
        # 如果是文件，直接判断是否 .pth
        if node.is_file():
            return node.suffix == ".pth"

        # 目录：深度用完则不再向下
        if depth_left is not None and depth_left <= 0:
            return False

        for child in _sorted_children(node):
            # 忽略项
            if _should_ignore(child, root, ignore_globs):
                continue
            # 符号链接目录根据开关决定是否深入
            if child.is_dir() and child.is_symlink() and not follow_symlinks:
                # 作为目录但不深入；是否算作包含 .pth？——不算（因为不展开）
                continue
            if contains_pth(child, None if depth_left is None else depth_left - 1):
                return True
        return False

    def print_branch(node: Path, prefix: str, is_last: bool, depth_left: Optional[int]) -> None:
        connector = "└── " if is_last else "├── "
        print(prefix + connector + node.name)

        # 文件：若是 .pth 就到此为止
        if node.is_file():
            return

        # 目录：需要继续向下打印“仍然通向 .pth 的孩子”
        if depth_left is not None and depth_left <= 0:
            return

        children = _sorted_children(node)
        # 过滤掉忽略项与不通向 .pth 的分支
        kept = []
        for ch in children:
            if _should_ignore(ch, root, ignore_globs):
                continue
            if ch.is_dir() and ch.is_symlink() and not follow_symlinks:
                continue
            if contains_pth(ch, None if depth_left is None else depth_left - 1):
                kept.append(ch)

        for i, ch in enumerate(kept):
            last = (i == len(kept) - 1)
            ext_prefix = prefix + ("    " if is_last else "│   ")
            print_branch(
                ch,
                ext_prefix,
                last,
                None if depth_left is None else depth_left - 1
            )

    # 顶层打印：根本身不加连接符，直接打印绝对路径；若根不含任何 .pth 则不输出任何内容
    if contains_pth(root, max_depth):
        print(root.resolve().as_posix())
        # 打印根下第一层（仅保留通向 .pth 的孩子）
        children = _sorted_children(root)
        kept = []
        for ch in children:
            if _should_ignore(ch, root, ignore_globs):
                continue
            if ch.is_dir() and ch.is_symlink() and not follow_symlinks:
                continue
            if contains_pth(ch, None if max_depth is None else max_depth - 1):
                kept.append(ch)

        for i, ch in enumerate(kept):
            print_branch(
                ch,
                prefix="",
                is_last=(i == len(kept) - 1),
                depth_left=None if max_depth is None else max_depth - 1
            )
    else:
        # 根不通向 .pth，则不打印任何东西
        pass


In [9]:
print_tree_only_pth(
    "/blue/shiboli.fsu/yifeisun.umich/adversarial_robustness_FNO/2D_NS_FNO2d_recurrent/saved_models/",
    show_files=True,
    max_depth=None,
    ignore_globs=[
        # 忽略某些目录或文件（可按名字或相对路径通配）
        ".git", "__pycache__", "*/wandb*", "**/logs/**",
        # 忽略非关心的文件后缀
        "*.gif","*.txt","*.png","*.ipynb","*.pkl","*.err","*.out","*.csv","*.sh","*.py"
    ],
    follow_symlinks=False,
)


/blue/shiboli.fsu/yifeisun.umich/adversarial_robustness_FNO/2D_NS_FNO2d_recurrent/saved_models
└── 2D
    ├── modes128_width40_epochs500_Tin10_T10
    │   └── NS_2d_FNO_model_trainedby_dim2d_nx256_N1150_solver=exponax_nu0.000_t20.0_train_all_frames.pth
    ├── modes12_width20_epochs500_Tin10_T10
    │   └── NS_2d_FNO_model_trainedby_dim2d_nx256_N1150_solver=exponax_nu0.000_t20.0_train_all_frames.pth
    ├── modes16_width60_epochs500_Tin10_T10
    │   └── NS_2d_FNO_model_trainedby_dim2d_nx256_N1150_solver=exponax_nu0.000_t20.0_train_all_frames.pth
    ├── modes32_width40_epochs500_Tin10_T10
    │   └── NS_2d_FNO_model_trainedby_dim2d_nx256_N1150_solver=exponax_nu0.000_t20.0_train_all_frames.pth
    ├── modes32_width40_epochs500_Tin1_T19
    │   └── NS_2d_FNO_model_trainedby_dim2d_nx256_N1150_solver=exponax_nu0.000_t20.0_train_all_frames.pth
    ├── modes32_width60_epochs500_Tin10_T10
    │   └── NS_2d_FNO_model_trainedby_dim2d_nx256_N1150_solver=exponax_nu0.000_t20.0_train_all_frames.pt

In [1]:
# 可在 Jupyter Notebook 里直接运行

from __future__ import annotations
from pathlib import Path
import os
import fnmatch
from typing import Iterable, List, Tuple, Optional, Callable

def _should_ignore(name: str, ignore_globs: Optional[List[str]]) -> bool:
    if not ignore_globs:
        return False
    return any(fnmatch.fnmatch(name, pat) for pat in ignore_globs)

def _sort_key(p: Path) -> Tuple[int, str]:
    # 目录优先，再按名称不区分大小写排序
    return (0 if p.is_dir() else 1, p.name.lower())

def _children(
    p: Path,
    show_files: bool,
    follow_symlinks: bool,
    ignore_globs: Optional[List[str]],
) -> List[Path]:
    try:
        items = list(p.iterdir())
    except PermissionError:
        return []

    # 过滤隐藏/忽略项
    filtered = []
    for q in items:
        name = q.name
        if name.startswith(".") and not show_files:
            # 如果完全不想显示隐藏文件/夹，可以把 next 行替换为: continue
            pass
        if _should_ignore(name, ignore_globs):
            continue
        if q.is_dir() or show_files:
            filtered.append(q)

    # 如果不跟随符号链接目录，则把符号链接目录当作叶子
    if not follow_symlinks:
        out = []
        for q in filtered:
            if q.is_dir() and q.is_symlink():
                # 仍然显示这个链接，但不深入
                out.append(q)
            else:
                out.append(q)
        filtered = out

    return sorted(filtered, key=_sort_key)

def print_tree(
    root: str | Path,
    *,
    show_files: bool = True,
    max_depth: Optional[int] = None,
    ignore_globs: Optional[List[str]] = None,
    follow_symlinks: bool = False,
    prefix: str = "",
) -> None:
    """
    打印文件树结构。

    参数
    ----
    root : 根目录路径
    show_files : 是否显示文件（为 False 则只显示目录）
    max_depth : 最大深度（None 表示不限制；0 表示只显示 root 本身）
    ignore_globs : 忽略的通配符列表，例如 ["*.pyc", ".git", "__pycache__"]
    follow_symlinks : 是否深入符号链接目录
    prefix : 递归时内部使用的绘制前缀
    """
    root = Path(root)
    if max_depth is not None and max_depth < 0:
        return

    # 根行
    if prefix == "":
        print(root.resolve().as_posix())

    children = _children(root, show_files=show_files, follow_symlinks=follow_symlinks, ignore_globs=ignore_globs)
    # 仅保留目录（当 show_files=False）或全部
    if not show_files:
        children = [c for c in children if c.is_dir()]

    # 分支绘制
    for i, child in enumerate(children):
        connector = "└── " if i == len(children) - 1 else "├── "
        print(prefix + connector + child.name)

        # 递归
        if child.is_dir() and (follow_symlinks or not child.is_symlink()):
            if max_depth is None or max_depth > 0:
                extension = "    " if i == len(children) - 1 else "│   "
                print_tree(
                    child,
                    show_files=show_files,
                    max_depth=(None if max_depth is None else max_depth - 1),
                    ignore_globs=ignore_globs,
                    follow_symlinks=follow_symlinks,
                    prefix=prefix + extension,
                )

print_tree("/blue/shiboli.fsu/yifeisun.umich/adversarial_robustness_FNO/2D_NS_FNO2d_recurrent/saved_models_kernel_abstracts/kernel_results", 
           show_files=True, max_depth=None, 
           ignore_globs=["*.gif","*.txt","*.png","*.ipynb","*.pkl","*.err","*.out","*.csv","*.sh","*.py"])


/blue/shiboli.fsu/yifeisun.umich/adversarial_robustness_FNO/2D_NS_FNO2d_recurrent/saved_models_kernel_abstracts/kernel_results
├── modes128_width40_epochs500_Tin10_T10.analysis.pt
├── modes12_width20_epochs500_Tin10_T10.analysis.pt
├── modes16_width60_epochs500_Tin10_T10.analysis.pt
├── modes32_width40_epochs500_Tin10_T10.analysis.pt
├── modes32_width40_epochs500_Tin1_T19.analysis.pt
├── modes32_width60_epochs500_Tin10_T10.analysis.pt
├── modes48_width40_epochs500_Tin10_T10.analysis.pt
├── modes48_width60_epochs500_Tin10_T10.analysis.pt
├── modes48_width80_epochs500_Tin10_T10.analysis.pt
├── modes64_width40_epochs500_Tin10_T10.analysis.pt
├── modes64_width60_epochs500_Tin10_T10.analysis.pt
├── modes64_width60_epochs500_Tin1_T19.analysis.pt
├── modes64_width80_epochs500_Tin10_T10.analysis.pt
└── modes96_width40_epochs500_Tin10_T10.analysis.pt
